# Projet NLP : Named entity recognition du titulaire / suppléant et de son affiliation politique

# Objectifs

On souhaite évaluer la performance d'une méthode Named-Entity-Recognition pour l'identification du titulaire de la profession de foi et de son affiliation politique. Pour cela, on tire parti des annotations disponibles, pour tagger les noms des titulaires / suppléants dans les transcriptions.

Plus précisément, on évalue ici la performance de GlinNER2 car : 
- c'est un modèle de type transformer de petite taille, uniquement CPU
- le modèle permet de faire du zero-shot learning et permet d'indiquer que l'on souhaite avoir l'affiliation politique correspondante à l'entité reconnue

# Annotations / création d'un jeu de test

On annote les professions de foi à partir des métadonnées disponibles.
Résulat : 
- liste des segments de texte où apparaissent le nom du titulaire / suppléant 
- affiliation politique de la profession de foi

## Lecture des métadonnées

In [89]:
import contextlib
import pandas as pd
import polars as pl
from tqdm import tqdm

In [130]:
# read with pandas, more permissive on CSV parsing
csv = pd.read_csv("archelect_search.csv")
csv['date'] = pd.to_datetime(csv['date'])

/tmp/ipykernel_2178/2974932302.py:1: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  csv = pd.read_csv("archelect_search.csv")


In [131]:
# select 1988 election (because we only have transcriptions for this election)
csv1988 = csv.loc[(csv['date'].dt.year==1988) & (csv['contexte-election']=='législatives')]

In [132]:
csv1988 = pl.from_pandas(csv1988)

In [163]:
# select only rows with non trivial titulaire
csv1988.get_column("titulaire-nom").value_counts().sort("count") # 79 valeurs "non mentionné"
csv1988.with_columns(
    pl.col("titulaire-nom").str.len_chars().alias("titulaire-nom-length")
).sort("titulaire-nom-length").select(pl.selectors.by_name("titulaire-nom","titulaire-nom-length")) # pas de chaîne vide
csv1988 = csv1988.filter(pl.col("titulaire-nom") != "non mentionné") # 3461 rows

## Lecture des transcripts

In [166]:
def read_transcript(id):
    try:
        with open("text_files/1988/legislatives/"+id+".txt", "r") as file:
            return file.read()
    except FileNotFoundError:
        return '' # transcript not found, return empty string

csv1988 = csv1988.with_columns(
    pl.col("id").map_elements(read_transcript, return_dtype=pl.String).alias("transcript")
)
csv1988.filter(pl.col("transcript") == '') # only 1 missing transcript
csv1988 = csv1988.filter(pl.col("transcript") != '') # 3460 rows

In [167]:
csv1988

id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,identifiant de circonscription,images,pdf,ocr_url,titulaire-nom,titulaire-prenom,titulaire-sexe,titulaire-age,titulaire-age-calcule,titulaire-age-tranche,titulaire-profession,titulaire-mandat-en-cours,titulaire-mandat-passe,titulaire-associations,titulaire-autres-statuts,titulaire-soutien,titulaire-liste,titulaire-decorations,suppleant-nom,suppleant-prenom,suppleant-sexe,suppleant-age,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,transcript
str,datetime[μs],str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""EL174_L_1988_06_001_01_1_PF_01""",1988-06-05 00:00:00,"""France;Assemblée Nationale;Ve …","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia803409.us.archive.or…","""https://ia803409.us.archive.or…","""https://ia803409.us.archive.or…","""Aulagne""","""Bernard""","""homme""","""45""","""45""","""entre 40 et 49 ans""","""marbrier""","""non mentionné""","""non mentionné""","""local""","""non mentionné""","""Front national""","""Liste d'entente populaire et n…","""non""","""Le Faucheur""","""Tiphaine""","""femme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Front national""","""Liste d'entente populaire et n…","""non""","""Élections Législatives du 5 Ju…"
"""EL174_L_1988_06_001_01_1_PF_02""",1988-06-05 00:00:00,"""Élections législatives;Assembl…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia803400.us.archive.or…","""https://ia803400.us.archive.or…","""https://ia803400.us.archive.or…","""Pujol""","""Régis""","""homme""","""58""","""58""","""entre 50 et 59 ans""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""chômeur""","""Comités Juquin""","""Une nouvelle politique à gauch…","""non""","""Mortel""","""Jean-François""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""Comités Juquin""","""Une nouvelle politique à gauch…","""non""","""RÉPUBLIQUE FRANÇAISE - Liberté…"
"""EL174_L_1988_06_001_01_1_PF_03""",1988-06-05 00:00:00,"""Assemblée Nationale;Ve Républi…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia801800.us.archive.or…","""https://ia801800.us.archive.or…","""https://ia801800.us.archive.or…","""Boyon""","""Jacques""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""conseiller général;maire""","""secrétaire d'Etat""","""non mentionné""","""non mentionné""","""Rassemblement pour la Républiq…","""Union du rassemblement et du c…","""oui""","""Morin""","""Paul""","""homme""","""non mentionné""","""non mentionné""","""non mentionné""","""non mentionné""","""conseiller général""","""maire-adjoint honoraire""","""non mentionné""","""non mentionné""","""Rassemblement pour la Républiq…","""Union du rassemblement et du c…","""oui""","""Sciences Po / fonds CEVIPOF EL…"
"""EL174_L_1988_06_001_01_1_PF_04""",1988-06-05 00:00:00,"""Assemblée Nationale;Ve Républi…","""Élections législatives de 1988…","""législatives""",1,"""EL174""","""01""","""Ain""","""01 - Ain""","""1""","""https://ia801807.us.archive.or…","""https://ia801807.us.archive.or…","""https://ia801807.us.archive.or…","""Saint-Pierre""","""Dominique""","""homme""","""47""","""47""","""entre 40 et 49 ans""","""avocat""","""conseiller régional""","""d

## Normalisation du nom

In [168]:
df = csv1988

In [173]:
df = df.with_columns(
    pl.col("transcript").str.to_lowercase()
      .str.normalize("NFKD")
      .str.replace_all(r"\p{CombiningMark}", "").alias("transcript-normalized"),
    pl.col("titulaire-nom").str.to_lowercase()
      .str.normalize("NFKD")
      .str.replace_all(r"\p{CombiningMark}", "").alias("titulaire-nom-normalized"),
)

In [174]:
df.select(pl.selectors.by_name("titulaire-nom","titulaire-nom-normalized"))

titulaire-nom,titulaire-nom-normalized
str,str
"""Aulagne""","""aulagne"""
"""Pujol""","""pujol"""
"""Boyon""","""boyon"""
"""Saint-Pierre""","""saint-pierre"""
"""Mornet""","""mornet"""
…,…
"""Hoarau""","""hoarau"""
"""Pihouee""","""pihouee"""
"""Hoarau""","""hoarau"""


## Positions du nom dans le transcript

In [176]:
import re

def find_aux(row):
    return [match.start() for match in re.finditer(row["pattern"], row["text"])]

df = df.with_columns(
     pl.struct(["transcript-normalized","titulaire-nom-normalized"])
        .struct.rename_fields(["text","pattern"])
        .map_elements(find_aux,return_dtype=pl.List(pl.Int64))
        .alias("titulaire-nom-positions")
)

In [178]:
df.select(pl.selectors.by_name("transcript","titulaire-nom-normalized","titulaire-nom-positions"))

transcript,titulaire-nom-normalized,titulaire-nom-positions
str,str,list[i64]
"""Élections Législatives du 5 Ju…","""aulagne""","[1240, 2866]"
"""RÉPUBLIQUE FRANÇAISE - Liberté…","""pujol""","[208, 3567]"
"""Sciences Po / fonds CEVIPOF EL…","""boyon""",[155]
"""Sciences Po / fonds CEVIPOF RÉ…","""saint-pierre""","[102, 1346]"
"""RÉPUBLIQUE FRANÇAISE - LIBERTÉ…","""mornet""",[84]
…,…,…
"""Claude HOARAU Égalité Responsa…","""hoarau""","[7, 846, 3701]"
"""Sciences Po / fonds CEVIPOF An…","""pihouee""",[2005]
"""Élie HOARAU Égalité Responsabi…","""hoarau""","[5, 3694]"


In [183]:
(df.select(pl.selectors.by_name("transcript","titulaire-nom","titulaire-nom-positions"))
    .filter(pl.col("titulaire-nom-positions").list.len() == 0)) # 50 rows where titulaire-nom does not appear in transcript

transcript,titulaire-nom,titulaire-nom-positions
str,str,list[i64]
"""ELECTIONS LEGISLATIVES DU 5 JU…","""Chevalier""",[]
"""DÉPARTEMENT DES ALPES-MARITIME…","""Vassalo""",[]
"""DONNEZ NOUS UN DEPUTE OM EN FI…","""Péron""",[]
"""Clément BESOMBES Conseiller Mu…","""Descombes""",[]
"""GEORGES CHAVANES Jean-Michel B…","""Chavanez""",[]
…,…,…
"""Sciences Po / fonds CEVIPOF Ma…","""Giovanelli""",[]
"""+ INITIATIVES COURAGE = RESULT…","""Guysel""",[]
"""Sciences Po / fonds CEVIPOF RE…","""Adevah-Poeuf""",[]


# Gliner2

## NER : reconnaissance du nom

In [49]:
from gliner2 import GLiNER2
extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")


/home/onyxia/work/nlpenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


NameError: name 'spacy' is not defined

In [50]:
import spacy
# from spacy.util import filter_spans
nlp = spacy.blank("fr")  # tokenizer only

In [51]:
schema = extractor.create_schema().entities({
    "PER": "person"
})


In [56]:
extractor.extract(df.item(0, "transcript"), schema, include_spans=True)

{'entities': {'PER': [{'text': 'Bernard AULAGNE', 'start': 1232, 'end': 1247},
   {'text': 'François MITTERRAND', 'start': 523, 'end': 542}]}}

In [82]:
pl.DataFrame({
    "text": ["apple pie", "banana split", "cherry tart"],
    "pattern": ["a", "split", "berry"]
}).with_columns(
    coords = pl.col("text").str.find_many(pl.col("pattern"))
)

/tmp/ipykernel_2178/2562633410.py:4: DeprecationWarning: `str.extract_many` with a flat string datatype is deprecated.
Please use `implode` to return to previous behavior.
See https://github.com/pola-rs/polars/issues/22149 for more information.
  }).with_columns(


text,pattern,coords
str,str,list[u32]
"""apple pie""","""a""",[0]
"""banana split""","""split""","[1, 3, … 7]"
"""cherry tart""","""berry""",[8]


In [88]:
def find_aux(row):
    return row["text"].find(row["pattern"])

pl.DataFrame({
    "text": ["apple pie", "banana split", "cherry tart"],
    "pattern": ["a", "split", "berry"]
}).with_columns(
    text_pattern = pl.struct(["text","pattern"])
    .map_elements(find_aux,return_dtype=pl.Int64)
)

text,pattern,text_pattern
str,str,i64
"""apple pie""","""a""",0
"""banana split""","""split""",7
"""cherry tart""","""berry""",-1


In [99]:
import re

def find_aux(row):
    return [match.start() for match in re.finditer(row["pattern"], row["text"])]

pl.DataFrame({
    "text": ["apple pie", "banana split", "cherry tart"],
    "pattern": ["a", "split", "berry"]
}).with_columns(
    text_pattern = pl.struct(["text","pattern"])
    .map_elements(find_aux,return_dtype=pl.List(pl.Int64))
)

text,pattern,text_pattern
str,str,list[i64]
"""apple pie""","""a""",[0]
"""banana split""","""split""",[7]
"""cherry tart""","""berry""",[]


In [114]:
import re

def find_aux(row):
    return [match.start() for match in re.finditer(row["pattern"], row["text"])]

pl.DataFrame({
    "toto": ["apple pie", "banana split", "cherry tart"],
    "tata": ["a", "split", "berry"]
}).with_columns(
    text_pattern = pl.struct(["toto","tata"])
        .struct.rename_fields(["text","pattern"])
        .map_elements(find_aux,return_dtype=pl.List(pl.Int64))
)

toto,tata,text_pattern
str,str,list[i64]
"""apple pie""","""a""",[0]
"""banana split""","""split""",[7]
"""cherry tart""","""berry""",[]
